In [1]:
import sys
import os
import pandas as pd
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.data_loading import consumers, accounts, transactions, category_mapping
from scripts.backfill_transactions import build_backfill_df
from scripts.feature_creation import create_all_features

df = build_backfill_df()
features_df = create_all_features(df, transactions, category_mapping)


FEATURE CREATION PIPELINE
Preparing daily data...
Daily data shape: (1183217, 6)
Creating balance features...
Balance features shape: (12900, 9)
Creating daily window features...
  30d: (12900, 9)
  60d: (12900, 9)
  90d: (12900, 9)
  180d: (12900, 9)
Creating transaction features...
Transaction features shape: (12900, 7)
Creating category features (top 30 categories)...
Category features (all-time): (14491, 60)
Category features (90d): (14491, 60)
Creating grouped category features...
Grouped category features shape: (14305, 5)
Creating overdraft & fee features...
Overdraft & fee features shape: (7219, 15)
Creating low balance risk features...


c:\Users\kangy\DSC180\dsc180a-prism-data\scripts\feature_creation.py:545: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  risk_all = rows_daily.groupby("prism_consumer_id").apply(
c:\Users\kangy\DSC180\dsc180a-prism-data\scripts\feature_creation.py:551: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  risk_30 = rows_daily.groupby("prism_consumer_id").apply(
c:\Users\kangy\DSC180\dsc180a-prism-data\scripts\feature_creation

Low balance risk features shape: (12900, 15)
Creating income regularity features...


c:\Users\kangy\DSC180\dsc180a-prism-data\scripts\feature_creation.py:676: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  income_reg = income_tx.groupby("prism_consumer_id").apply(calc_income_regularity)
c:\Users\kangy\DSC180\dsc180a-prism-data\scripts\feature_creation.py:677: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  paycheck_reg = paycheck_tx.groupby("prism_consumer_id").apply(calc_paycheck_regularity)


Income regularity features shape: (14035, 7)
Creating paycheck-to-paycheck features...
Paycheck-to-paycheck features shape: (12900, 3)

Joining all features...

FINAL FEATURE MATRIX
Shape: (12900, 218)
Number of features: 217
Number of consumers: 12900


In [7]:
# save features to csv
features_df.to_csv('features.csv', index=False)

In [2]:
'DQ_TARGET' in features_df.columns

True

# Data Preparation

In [3]:
# keep only labeled rows
df = features_df[features_df["DQ_TARGET"].notna()].copy()

X = df.drop(columns=["DQ_TARGET"])
y = df["DQ_TARGET"].astype(int)

## Train Test Split

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Prepare Data
# ---------------------------------------------------------
# Ensure no infinite values
df = features_df.replace([np.inf, -np.inf], np.nan)
df = df[df["DQ_TARGET"].notna()].copy()

X = df.drop(columns=["DQ_TARGET"]).select_dtypes(include=[np.number]).fillna(0)
y = df["DQ_TARGET"].astype(int)

# 2. Create Train / Validation / Test Split (60% / 20% / 20%)
# ---------------------------------------------------------
# First Split: Separate out the Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second Split: Separate the remaining 80% into Train (60%) and Val (20%)
# (0.25 of the remaining 80% = 20% of the total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Data Shapes:")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")


Data Shapes:
Train: (6189, 217) | Val: (2064, 217) | Test: (2064, 217)


# Recursive Feature Elimination (RFE)

## Logistic Regression

In [8]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("StandardScaler Done.")

# RFE
model = LogisticRegression(
    solver='liblinear',
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

rfe = RFE(estimator=model, n_features_to_select=50, step=1)

print("Running RFE...")
rfe.fit(X_train_scaled, y_train)


StandardScaler Done.
Running RFE...


,estimator,LogisticRegre...r='liblinear')
,n_features_to_select,50
,step,1
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1


In [9]:
from sklearn.metrics import roc_auc_score, accuracy_score

# Evaluate Performance (Accuracy & AUC)
# ---------------------------------------------------------
selected_cols = X_train.columns[rfe.support_]
print(f"\nTop {len(selected_cols)} Features Selected: {selected_cols.tolist()}")

# Refit model on selected features
model.fit(X_train_scaled[selected_cols], y_train)

# Predict (Probabilities for AUC, Class Labels for Accuracy)
y_train_prob = model.predict_proba(X_train_scaled[selected_cols])[:, 1]
y_val_prob   = model.predict_proba(X_val_scaled[selected_cols])[:, 1]

y_train_pred = model.predict(X_train_scaled[selected_cols])
y_val_pred   = model.predict(X_val_scaled[selected_cols])

# Calculate Metrics
train_auc = roc_auc_score(y_train, y_train_prob)
val_auc   = roc_auc_score(y_val, y_val_prob)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc   = accuracy_score(y_val, y_val_pred)

print(f"\n--- Performance Report ---")
print(f"Train AUC:      {train_auc:.4f} | Train Accuracy: {train_acc:.4f}")
print(f"Validation AUC: {val_auc:.4f} | Val Accuracy:   {val_acc:.4f}")

# Diagnostic Check
# ---------------------------------------------------------
gap = train_auc - val_auc
print(f"Gap (Train - Val): {gap:.4f}")

if gap > 0.05:
    print("⚠️ Warning: Possible Overfitting. Consider reducing features or increasing regularization (C).")
elif train_auc < 0.60:
    print("⚠️ Warning: Possible Underfitting. Model may be too simple or features are not predictive.")
else:
    print("✅ Model looks healthy.")


Top 50 Features Selected: ['balance__mean__all', 'balance__median__all', 'balance__min__all', 'balance__max__all', 'balance__std__all', 'n_days__all', 'balance__mean__30d', 'balance__min__30d', 'balance__std__30d', 'cashflow__net__30d', 'cashflow__mean_daily__30d', 'cashflow__volatility__30d', 'n_tx__30d', 'balance__min__60d', 'cashflow__net__60d', 'cashflow__mean_daily__60d', 'cashflow__volatility__60d', 'n_tx__60d', 'balance__mean__90d', 'cashflow__mean_daily__90d', 'cashflow__volatility__90d', 'n_days__90d', 'balance__min__180d', 'cashflow__mean_daily__180d', 'cashflow__volatility__180d', 'n_tx__180d', 'n_days__180d', 'tx__n__all', 'tx__std_amount__all', 'tx__max_credit__all', 'cat_0__cat_net_total__all', 'cat_1__cat_net_total__all', 'cat_6__cat_net_total__all', 'cat_22__cat_net_total__all', 'cat_46__cat_net_total__all', 'cat_0__cat_net_total__90d', 'cat_6__cat_net_total__90d', 'cat_14__cat_net_total__90d', 'cat_22__cat_net_total__90d', 'cat_46__cat_net_total__90d', 'cat_3__cat_n__

## Random Forest

In [10]:
from sklearn.feature_selection import RFE
# random forest for feature importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("StandardScaler Done.")

# RFE
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

rfe = RFE(estimator=model, n_features_to_select=50, step=1)

print("Running RFE...")
rfe.fit(X_train_scaled, y_train)


StandardScaler Done.
Running RFE...


,estimator,RandomForestC...ndom_state=42)
,n_features_to_select,50
,step,1
,verbose,0
,importance_getter,'auto'
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [11]:
from sklearn.metrics import roc_auc_score, accuracy_score

# Evaluate Performance (Accuracy & AUC)
# ---------------------------------------------------------
selected_cols = X_train.columns[rfe.support_]
print(f"\nTop {len(selected_cols)} Features Selected: {selected_cols.tolist()}")

# Refit model on selected features
model.fit(X_train_scaled[selected_cols], y_train)

# Predict (Probabilities for AUC, Class Labels for Accuracy)
y_train_prob = model.predict_proba(X_train_scaled[selected_cols])[:, 1]
y_val_prob   = model.predict_proba(X_val_scaled[selected_cols])[:, 1]

y_train_pred = model.predict(X_train_scaled[selected_cols])
y_val_pred   = model.predict(X_val_scaled[selected_cols])

# Calculate Metrics
train_auc = roc_auc_score(y_train, y_train_prob)
val_auc   = roc_auc_score(y_val, y_val_prob)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc   = accuracy_score(y_val, y_val_pred)

print(f"\n--- Performance Report ---")
print(f"Train AUC:      {train_auc:.4f} | Train Accuracy: {train_acc:.4f}")
print(f"Validation AUC: {val_auc:.4f} | Val Accuracy:   {val_acc:.4f}")

# Diagnostic Check
# ---------------------------------------------------------
gap = train_auc - val_auc
print(f"Gap (Train - Val): {gap:.4f}")

if gap > 0.05:
    print("⚠️ Warning: Possible Overfitting. Consider reducing features or increasing regularization (C).")
elif train_auc < 0.60:
    print("⚠️ Warning: Possible Underfitting. Model may be too simple or features are not predictive.")
else:
    print("✅ Model looks healthy.")


Top 50 Features Selected: ['balance__mean__all', 'balance__median__all', 'balance__min__all', 'balance__max__all', 'balance__std__all', 'balance__pct_below_500__all', 'n_days__all', 'balance__mean__30d', 'balance__min__30d', 'balance__std__30d', 'n_tx__30d', 'balance__mean__60d', 'balance__min__60d', 'cashflow__mean_daily__60d', 'n_tx__60d', 'balance__mean__90d', 'balance__min__90d', 'cashflow__mean_daily__90d', 'n_tx__90d', 'balance__mean__180d', 'balance__min__180d', 'n_days__180d', 'tx__std_amount__all', 'credit__total__all', 'cat_0__cat_net_total__all', 'cat_1__cat_net_total__all', 'cat_4__cat_net_total__all', 'cat_13__cat_net_total__all', 'cat_14__cat_net_total__all', 'cat_17__cat_net_total__all', 'cat_18__cat_net_total__all', 'cat_20__cat_net_total__all', 'cat_0__cat_n__all', 'cat_1__cat_n__all', 'cat_1__cat_net_total__90d', 'cat_4__cat_net_total__90d', 'cat_16__cat_net_total__90d', 'cat_19__cat_net_total__90d', 'cat_20__cat_net_total__90d', 'cat_26__cat_net_total__90d', 'cat_1_